In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np


# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    transforms.Resize((28, 28)),
    transforms.Grayscale(3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset  = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")



In [ ]:
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
# Write your code here

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)


imgs, labels = next(iter(train_loader))

mean = torch.tensor([0.485, 0.456, 0.406]).view(1,3,1,1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(1,3,1,1)

imgs = imgs * std + mean
imgs = imgs.clamp(0, 1)

plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i+1)
    img = imgs[i].permute(1, 2, 0)
    plt.imshow(img)
    plt.title(letters[labels[i]-1])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here

device = "cuda" if torch.cuda.is_available() else "cpu"

model = efficientnet_v2_s(weights="DEFAULT")

# freeze backbone
for p in model.features.parameters():
    p.requires_grad = False

# replace classifier head
in_f = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_f, num_classes)

model = model.to(device)

print(model.classifier)


In [ ]:
# Write your code here
def train_one_epoch(model, loader, loss_fn, opt, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device)
        labels = (labels - 1).to(device)

        opt.zero_grad()
        out = model(imgs)
        loss = loss_fn(out, labels)
        loss.backward()
        opt.step()

        total_loss += loss.item() * imgs.size(0)
        preds = out.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        if (i+1) % 100 == 0:
            print("step", i+1, "loss", loss.item())

    return total_loss / total, correct / total


def validate_one_epoch(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = (labels - 1).to(device)  # IMPORTANT

            out = model(imgs)
            loss = loss_fn(out, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_loss = total_loss / total
    acc = correct / total

    return avg_loss, acc


In [ ]:
# Write your code here
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(device)
loss_fn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(epochs):
    tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, opt, device)
    va_loss, va_acc = validate_one_epoch(model, test_loader, loss_fn, device)

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    train_accs.append(tr_acc)
    val_accs.append(va_acc)

    print(f"Epoch {epoch+1}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.4f} | val loss {va_loss:.4f} acc {va_acc:.4f}")

plt.figure()
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.show()

plt.figure()
plt.plot(train_accs, label="train")
plt.plot(val_accs, label="val")
plt.xlabel("epoch")
plt.ylabel("acc")
plt.legend()
plt.show()

In [ ]:
# Write your code here
def validate_one_epoch_tta(model, loader, loss_fn, device):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = (labels - 1).to(device)

            out1 = model(imgs)

            h_imgs = torch.flip(imgs, dims=[3])  # horizontal flip (W)
            out2 = model(h_imgs)

            v_imgs = torch.flip(imgs, dims=[2])  # vertical flip (H)
            out3 = model(v_imgs)

            out = (out1 + out2 + out3) / 3.0

            loss = loss_fn(out, labels)

            total_loss += loss.item() * imgs.size(0)
            preds = out.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total


In [ ]:
tta_loss, tta_acc = validate_one_epoch_tta(model, test_loader, loss_fn, device)
print("TTA val loss:", tta_loss, "TTA val acc:", tta_acc)
